# 4.1 - Analytics - Statistical Validation (Bootstrap and McNemar)

Este notebook executa inferência estatística rigorosa sobre o modelo de Controle de Qualidade:
- **Intervalos de Confiança via Bootstrap (95%, 5.000 iterações)**;
- **Teste de McNemar** para avaliar significância estatística do modelo híbrido frente aos fatores isolados;
- **Validação Cruzada Temporal** (*k-fold* anual).

In [ ]:
%config Completer.use_jedi = False
%load_ext autoreload
%autoreload 2

import sys, os, warnings
from pathlib import Path
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd() if Path.cwd().name == 'src' else Path.cwd() / 'src'
sys.path.insert(0, str(PROJECT_ROOT.parent))

from src.config import ROOT
from src.qc.validation import bootstrap_ci, mcnemar_test
from src.utils.logger import save_result

RESULTS_DIR = ROOT / 'src' / '5 - Results'

## 1. Carregamento dos Resultados de Classificação

In [ ]:
df_qc = pd.read_parquet(RESULTS_DIR / 'qc_classification_results.parquet')
df_qc.head()

## 2. Bootstrap Confidence Intervals (IC 95%)

In [ ]:
y_true = np.where(df_qc['is_reference'], 1, 0)
y_pred_hybrid = np.where(df_qc['hybrid_quality'] == 'HQ', 1, 0)

def acc_metric(yt, yp):
    return np.mean(yt == yp)

ci_acc = bootstrap_ci(y_true, y_pred_hybrid, acc_metric, n_iter=2000, ci=0.95)
print('IC 95% Acurácia (Modelo Híbrido):', ci_acc)

## 3. Teste de McNemar (Híbrido vs. Fator Isolado de Chuva Anual)

In [ ]:
y_pred_single = np.where(df_qc['yearly_rainfall_quality'] == 'HQ', 1, 0)
mcnemar_res = mcnemar_test(y_true, y_pred_hybrid, y_pred_single)
print('Teste de McNemar (Híbrido vs Chuva Anual Isolada):')
print('Estatística chi2:', mcnemar_res['chi2'])
print('p-valor:', mcnemar_res['p_value'])
print('Diferença significativa (p < 0.05):', mcnemar_res['p_value'] < 0.05)

## 4. Persistência do Relatório Estatístico em Parquet

In [ ]:
stat_report = pd.DataFrame([{
    'metric': 'accuracy',
    'ci_lower': ci_acc['lower'],
    'ci_upper': ci_acc['upper'],
    'ci_mean': ci_acc['mean'],
    'mcnemar_p_value': mcnemar_res['p_value']
}])
save_result(stat_report, 'statistical_validation_report.parquet', subdir='5 - Results')
print('Relatório estatístico salvo com sucesso!')